# Week 8 — Machine Learning for Astronomical Inference

**Contact:** 3 hours lecture  
**Course:** MSc Astrostatistics for the Survey Era

## Learning notebooks from the Crete school
- `../../05_ML/01-ML_Intro_and_Clustering.ipynb`
- `../../05_ML/02-ML_Classification.ipynb`

These Crete notebooks are **learning material**. This lecture notebook supplies the semester narrative, deeper astronomical interpretation, and the binary-SMBH case study.

## Three-hour pacing
- **0:00–0:20** — recap, motivating scientific question, diagnostic poll
- **0:20–1:05** — theory block I + worked derivation
- **1:05–1:15** — short break / discussion
- **1:15–2:00** — theory block II + computational demonstration
- **2:00–2:10** — short break
- **2:10–2:45** — binary-SMBH case study / failure mode
- **2:45–3:00** — synthesis, exit questions, bridge to the 4-hour practical

## Learning objectives

- Translate an astronomical question into a supervised or unsupervised ML task.
- Choose data splits that respect independent simulation units.
- Establish a scientific baseline before fitting flexible models.
- Use appropriate regression/classification metrics and residual diagnostics.
- Interpret feature importance cautiously and distinguish prediction from causal explanation.

## 1. ML begins with the scientific target


Machine learning maps inputs \(x\) to outputs \(y\), but a scientifically useful model begins before the algorithm. We must define what \(x\) represents observationally, whether \(y\) would be known for training in real applications, and what population the training pairs were sampled from.

A classifier for a simulator mechanism is a controlled exercise. A predictor of a latent physical parameter from light-curve summaries is closer to an inverse problem. These tasks require different evaluation and have different simulation-to-real risks.

> **Discussion checkpoint.** Identify the estimand, observed data, and strongest modelling assumption in the example just discussed.

### Worked computational demonstration — Baseline regression

Run the cell, then change one assumption and explain what changes statistically.

In [ ]:
import sys, numpy as np
sys.path.append('../src')
from course_utils import load_scalar_sample
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

df=load_scalar_sample()
features=['inputs__redshift','inputs__eccentricity','inputs__Eddington_fraction']
X_reg=df[features].to_numpy()
y_reg=np.log10(df['inputs__period'].to_numpy())
Xtr_reg,Xte_reg,ytr_reg,yte_reg=train_test_split(X_reg,y_reg,test_size=.25,random_state=2026)
baseline_reg=np.full_like(yte_reg,np.mean(ytr_reg))
lr=LinearRegression().fit(Xtr_reg,ytr_reg)
pred_reg=lr.predict(Xte_reg)
print('baseline RMSE',mean_squared_error(yte_reg,baseline_reg)**.5)
print('linear RMSE',mean_squared_error(yte_reg,pred_reg)**.5)

## 2. Data splitting and leakage


Random row splitting assumes rows are independent and identically distributed. If several rows share a simulation seed, object, host or derived realisation, random splitting leaks information across train and test sets.

Feature leakage is even more direct: a derived feature may encode the target through construction. In this course, any `inputs__...` truth column is forbidden when the goal is to infer that physical parameter from observables.

A test set is a scientific resource. It should represent the population on which the claim is made and should not be repeatedly consulted during model development.

### Worked computational demonstration — Classification metrics example

Run the cell, then change one assumption and explain what changes statistically.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
target=df['inputs__Doppler_boosting'].astype(int).to_numpy()
X_cls=df[['inputs__inclination','inputs__mass_ratio','inputs__redshift']].to_numpy()
Xtr_cls,Xte_cls,ytr_cls,yte_cls=train_test_split(X_cls,target,test_size=.25,random_state=2026,stratify=target)
clf=LogisticRegression(max_iter=2000).fit(Xtr_cls,ytr_cls)
prob_cls=clf.predict_proba(Xte_cls)[:,1]
print('ROC AUC',roc_auc_score(yte_cls,prob_cls))
print(classification_report(yte_cls,prob_cls>.5))

## 3. Baselines, bias and variance


A flexible model should beat an interpretable baseline under the metric that matters. For regression, useful baselines include predicting the training mean/median or a simple linear model. For classification, compare with prevalence, logistic regression or a simple threshold if physically motivated.

The bias-variance trade-off is not eliminated by large neural networks. A model can have low training error and poor generalisation. Cross-validation estimates variability across training partitions but does not protect against domain shift if every fold comes from the same simulation distribution.

### Worked computational demonstration — Residual audit

Run the cell, then change one assumption and explain what changes statistically.

In [ ]:
res_reg=pred_reg-yte_reg
print('bias',res_reg.mean(),'RMSE',np.sqrt(np.mean(res_reg**2)))
# Students: plot regression residuals against each regression feature and interpret structure.

## 4. Metrics and scientific diagnostics


Regression metrics such as RMSE and MAE emphasise different aspects of the residual distribution. Correlation can be high even with severe bias. Always inspect residuals against target, redshift, S/N, cadence and other covariates.

Classification accuracy is misleading with imbalanced classes. Precision, recall, ROC-AUC and precision-recall curves answer different questions. If predicted probabilities will be used scientifically, calibration is as important as discrimination.

Metrics should be paired with plots that expose where the model fails.

## 5. Feature importance is not physics


Tree importance, permutation importance and SHAP-like attributions describe the predictive model under its data distribution. They do not prove causal influence. A feature can be important because it acts as a proxy for survey selection or simulator design.

For the SMBH data, sky field or cadence features might predict a label if simulation construction accidentally couples them. A scientifically literate analysis treats such success as a potential bug, not automatically as a discovery.

## 6. Unsupervised learning and representation


Clustering can reveal structure but every clustering algorithm imposes a geometry. Standardisation, distance metric and dimensionality reduction can radically alter the groups found.

In astronomy, clusters should be validated against stability and scientific meaning, not accepted because a 2D embedding looks separated. This caution carries directly into learned latent representations and foundation-style models.

## Deep dive — pipelines, preprocessing and cross-validation

Preprocessing must be learned from training data only. Suppose features are standardised by

\[
x'=\frac{x-\hat\mu}{\hat\sigma}.
\]

If \(\hat\mu,\hat\sigma\) are calculated using the test set, information from the test distribution enters training. Software pipelines help ensure transformations are fit separately inside each training fold.

### Cross-validation answers a conditional question

K-fold cross-validation estimates performance when future data are exchangeable with the same distribution and the same grouping structure. Grouped data require GroupKFold-like logic. Time-dependent prediction may require forward-chaining rather than random folds.

Cross-validation does not estimate performance under a different survey or redshift regime unless the split explicitly represents that shift.

### Class imbalance and decision thresholds

A classifier outputs scores/probabilities; the decision threshold should depend on scientific costs. If follow-up time is expensive, high precision may matter. If missing rare transients is costly, recall may dominate.

ROC-AUC averages over thresholds and can look strong in severe imbalance. Precision-recall curves are often more informative when positives are rare.

### Physical baselines

A baseline can be scientifically motivated. If a mechanism is expected to depend strongly on inclination, a one-variable logistic model is an excellent comparator. A neural network should justify its complexity by improved validated performance, calibrated uncertainty, or richer scientifically useful output.

## Binary-SMBH synthesis questions

Answer verbally before leaving the lecture.

1. Which quantity in today's topic is a **property of the simulator**, and which could be an **observable property of a real survey**?
2. What uncertainty or selection effect would most strongly distort the result?
3. What diagnostic would you require before trusting the inference?
4. What information would be unavailable if these were real LSST sources rather than simulations?

## Key equations / ideas to retain

Create your own 6–10 line summary here during class.

## Further reading

- Crete `05_ML` notebooks.
- Hastie, Tibshirani & Friedman, *Elements of Statistical Learning*.
- Ivezić et al., ML chapters.

# Part II — Extended lecture development

## 13. Bias–variance decomposition

For squared-error prediction, expected test error can be understood through
bias, variance and irreducible noise. A very rigid model can underfit and have
high bias. A highly flexible model can fit training fluctuations and have high
variance.

The practical implication is that training error almost always decreases with
model complexity, while validation error may turn upward.

Cross-validation estimates this generalisation trade-off within the sampled
population; it does not automatically protect against distribution shift.

## 14. Hyperparameter tuning is part of the model-selection procedure

Tree depth, SVM \(C\), kernel scale, number of neighbours and neural-network
architecture are not fitted in the same way as ordinary model parameters, but
choosing them using validation data still adapts the analysis to those data.

Therefore a final test set should remain untouched after hyperparameter tuning.

Nested cross-validation can estimate performance when model selection itself
must be included inside the evaluation loop.

## 15. PCA and dimensionality reduction

PCA finds orthogonal directions maximising variance. If centred data matrix is
\(X\), PCA diagonalises the covariance matrix and orders eigenvectors by
explained variance.

PCA is unsupervised: the direction with the largest variance is not necessarily
the direction most useful for predicting the target.

Scaling matters enormously. If one feature is numerically measured in millions
and another in fractions, unscaled PCA can be dominated by units rather than
scientific structure.

The Crete ML-practices notebook develops this as a preprocessing tool.

In [ ]:
# PCA on transformed SMBH inputs
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X_pca=pd.DataFrame({
    "logM":np.log10(df["inputs__total_mass"].astype(float)),
    "q":df["inputs__mass_ratio"].astype(float),
    "logP":np.log10(df["inputs__period"].astype(float)),
    "e":df["inputs__eccentricity"].astype(float),
    "z":df["inputs__redshift"].astype(float),
    "logfEdd":np.log10(df["inputs__Eddington_fraction"].astype(float)),
})
Xp=StandardScaler().fit_transform(X_pca)
pca=PCA().fit(Xp)
print("explained variance ratios:",pca.explained_variance_ratio_)
print("cumulative:",np.cumsum(pca.explained_variance_ratio_))

## 16. K-means objective and scaling sensitivity

K-means assumes Euclidean squared distance is meaningful. If feature \(x_j\)
is multiplied by 100, its contribution to distance is multiplied by \(10^4\).

Thus clustering can change simply because units change.

Standardisation is common, but even that encodes a choice: every feature is
given equal marginal variance. If physical distance should weight features
differently, a domain-specific metric may be preferable.

Clusters are outputs of an algorithm/geometry, not discovered natural kinds by
default.

## 17. DBSCAN density assumptions

DBSCAN declares points core points if at least `min_samples` neighbours lie
within radius \(\epsilon\). Connected core regions form clusters.

Advantages:
- no need to specify \(K\);
- arbitrary cluster shapes;
- explicit noise points.

Limitations:
- sensitive to feature scale;
- one \(\epsilon\) struggles with variable-density clusters;
- high-dimensional distances concentrate.

For astronomical catalogue work, DBSCAN can be useful for anomaly/group
exploration, but its clusters require stability and physical validation.

## 18. SVM margins and the kernel trick

For binary labels \(y_i\in\{-1,+1\}\), a linear soft-margin SVM balances a wide
margin against violations. The hinge loss is

\[
L_i=\max[0,1-y_i f(x_i)].
\]

Kernel methods replace inner products with \(K(x,x')\), implicitly allowing
nonlinear feature spaces without explicitly constructing them.

This is powerful but can hide scale sensitivity. RBF kernels require thoughtful
feature standardisation and hyperparameter tuning.

## 19. Tree impurity and Random Forest averaging

For class proportions \(p_k\), Gini impurity is

\[
G=1-\sum_k p_k^2.
\]

A tree chooses splits reducing impurity. A random forest averages decorrelated
trees trained with bootstrap samples and random feature subsets.

The forest often provides a strong tabular baseline, but:
- impurity importance favours some feature types;
- probability calibration can be poor;
- extrapolation in regression is limited;
- correlated features share/compete for importance.

Permutation importance on held-out data is usually easier to interpret as a
predictive diagnostic.

In [ ]:
# Permutation importance example
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

features=["inputs__inclination","inputs__mass_ratio",
          "inputs__eccentricity","inputs__redshift"]
Xc=df[features].astype(float)
yc=df["inputs__Doppler_boosting"].astype(int)

Xtr,Xte,ytr,yte=train_test_split(
    Xc,yc,test_size=.25,random_state=2026,stratify=yc)
rf=RandomForestClassifier(n_estimators=200,random_state=2026,n_jobs=-1).fit(Xtr,ytr)
pi=permutation_importance(rf,Xte,yte,n_repeats=10,random_state=2026)
display(pd.DataFrame({"feature":features,"importance":pi.importances_mean})
        .sort_values("importance",ascending=False))

## 20. Baseline hierarchy

A useful prediction study can use a ladder:

1. constant/prevalence baseline;
2. simple physically motivated one-feature model;
3. linear/logistic multivariate model;
4. tree/kernel model;
5. neural model only if needed.

At every step ask what new scientific capability the complexity buys:
lower error, better calibration, nonlinear interaction, richer posterior or
robustness.

This prevents "deep learning by default."

## 21. Error analysis is often more valuable than the leaderboard

Sort validation examples by absolute error and inspect:
- redshift;
- target value;
- mechanism flags;
- cadence/SNR;
- distance from training support.

Failures can reveal missing features, non-identifiability or data-generation
structure.

A mean RMSE collapses all of this into one number.

## 22. Classification threshold as a scientific decision

Predicted probability \(p\) becomes a class only after choosing a threshold.

The optimal threshold depends on costs:
- false-positive follow-up cost;
- false-negative missed discovery cost;
- telescope allocation;
- desired catalogue purity/completeness.

Thus classification naturally leads to Week-11 decision theory. The statistical
model supplies uncertainty; the scientific programme chooses an action.

## 23. A valid SMBH feature set

For an inverse task, acceptable features should be obtainable from blinded
observables: e.g. cadence counts, median magnitudes, variability amplitudes,
periodogram summaries or learned light-curve representations.

Forbidden features include the target itself and hidden simulator truth.

A model trained on physical input variables can still be useful for a
methodological demonstration, but it must not be presented as an observational
inverse model.

# Part III — Worked ML design problems

## Problem A — a suspiciously perfect model

Suppose a classifier reaches 99.99% accuracy. Before celebrating, audit:
- is `id` included?
- is target truth included directly?
- is a derived feature mathematically constructed from the target?
- do train/test share simulations?
- does a field label encode the target through data construction?

Extremely strong performance is sometimes evidence of a problem, not evidence
of a breakthrough.

## Problem B — accuracy under class imbalance

If only 2% of objects are positive, an always-negative classifier has 98%
accuracy.

Its recall for the positive class is zero.

Ask students to calculate precision, recall and expected follow-up yield for two
thresholds. This converts abstract metrics into observing consequences.

## Problem C — regression to the mean

Flexible regressors often underpredict extreme high targets and overpredict
extreme low targets, especially when data are sparse at the edges.

Plot residual versus true target. A slope pattern can reveal shrinkage toward
the training mean even when overall RMSE looks respectable.

This is particularly relevant if rare extreme binaries are scientifically the
most interesting objects.